# 装饰器

学习目标：能编写保留参数、返回值和异常行为的函数装饰器，判断装饰与调用顺序，并检查包装后的元数据。

前置知识：函数对象、闭包、参数解包、名称绑定、实例方法与 self、异常传播和 try/finally。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 函数对象与包装函数

### 1.1 传递函数，不是提前调用

想在问候函数运行前后增加提示，可以把函数交给另一个函数处理。函数本身也是对象：greet 表示函数对象，greet("小林") 才表示调用它并取得返回值。

函数名只是引用对象的名称。下面先保存一个别名，后面用它比较装饰前后的入口。

In [1]:
def greet(name):
    """返回给指定姓名的问候语。"""
    return f"你好，{name}"


original_greet = greet
print(original_greet is greet)  # True：赋值没有复制或执行函数。
print(original_greet("小林"))  # 你好，小林

True
你好，小林


### 1.2 用闭包保存被包装函数

包装函数（wrapper function）在内部调用另一个函数，并可以在调用前后增加操作。装饰器（decorator）接收被装饰函数，返回将要绑定到原函数名的对象；本章主要返回一个包装函数。

下面 announce 接收 func，func 表示被包装的函数；内部的 wrapper 接收问候时的 name。wrapper 使用外层的 func，形成闭包，因此 announce 返回后仍能调用它。这承接闭包的用法，本章关注它如何保存被包装函数。

return wrapper 交回函数对象；这里不能写成 wrapper()，否则会立即调用它。下面沿用 greet。

In [2]:
def announce(func):
    """为接收姓名的函数增加开始与结束提示。"""

    def wrapper(name):
        """调用已保存的函数并返回其结果。"""
        print("开始问候")
        result = func(name)
        print("结束问候")
        return result

    return wrapper


wrapped_greet = announce(greet)  # 此时只创建包装函数，不打印提示。
print(wrapped_greet is greet)  # False：包装函数是另一个对象。
print(wrapped_greet("小林"))  # 开始问候、结束问候、你好，小林，各占一行。

False
开始问候
结束问候
你好，小林


## 2 @ 语法与名称重新绑定

在函数定义前写 @announce，会把新定义的函数交给 announce，再把返回对象绑定到该函数名。它近似于先定义函数，再执行 greet = announce(greet)。

这个等价写法用于理解绑定过程：真正的 @ 语法不会先把未装饰函数临时绑定到 greet。装饰器也不必总创建包装函数，它可以返回收到的函数；具体行为由装饰器实现决定。

下面沿用 announce 和 original\_greet。原先保存的别名仍指向原函数，名称重新绑定不会追溯修改已有引用。

In [3]:
@announce
def greet(name):
    """返回给指定姓名的问候语。"""
    return f"你好，{name}"


def manual_greet(name):
    """返回给指定姓名的问候语。"""
    return f"你好，{name}"


manual_greet = announce(manual_greet)
print(greet("小周"))  # 开始问候、结束问候、你好，小周。
print(manual_greet("小周"))  # 与上一次相同，展示显式绑定写法。
print(original_greet("小周"))  # 只有你好，小周：旧引用没有经过包装。

开始问候
结束问候
你好，小周
开始问候
结束问候
你好，小周
你好，小周


## 3 定义时装饰，调用时运行

### 3.1 分清装饰器函数体与原函数体

执行带装饰器的 def 语句时，就会求值装饰器表达式并应用装饰器。通常，此时创建包装函数；以后调用函数名，才进入包装函数和原函数体。

这是下面实现的行为，不是 Python 禁止装饰器在定义时调用原函数。原函数何时运行，仍取决于装饰器是否调用它。

In [4]:
def show_phase(func):
    """打印应用装饰器和进入包装函数的时机。"""
    print("应用装饰器")

    def wrapper():
        """显示调用阶段并转交执行。"""
        print("进入包装函数")
        return func()

    return wrapper


@show_phase
def read_title():
    """显示原函数开始执行并返回标题。"""
    print("运行原函数")
    return "装饰器"


print("定义结束")
print(read_title())
print(read_title())
# 先输出应用装饰器、定义结束；每次调用才输出进入包装函数、
# 运行原函数、装饰器。两次调用不会再次应用装饰器。

应用装饰器
定义结束
进入包装函数
运行原函数
装饰器
进入包装函数
运行原函数
装饰器


### 3.2 每次执行定义语句都会再次装饰

“定义时执行”不等于“整个程序只执行一次”。如果带装饰器的 def 位于另一个函数内部，每次调用外层函数并执行到该定义，都会重新创建和装饰函数。

下面沿用 show\_phase。build\_reader 负责创建函数，返回后的调用才读取标题。

In [5]:
def build_reader():
    """每次调用都创建一个新的标题读取函数。"""

    @show_phase
    def reader():
        """返回固定的章节标题。"""
        return "装饰器"

    return reader


first_reader = build_reader()
second_reader = build_reader()
# 前两次构建各输出一次应用装饰器。
print(first_reader is second_reader)  # False：得到两个不同的包装函数。
print(first_reader())  # 进入包装函数、装饰器；不会重新应用装饰器。

应用装饰器
应用装饰器
False
进入包装函数
装饰器


## 4 保留参数、返回值与异常行为

### 4.1 原样转发位置与关键字实参

只接收 name 的包装函数不能通用于不同参数列表。wrapper(\*args, \*\*kwargs) 中，args 收集位置实参组成的元组，kwargs 收集关键字实参组成的字典；func(\*args, \*\*kwargs) 再把它们解包传给被包装函数。

通用包装函数能接收这些实参，不代表原函数允许任意调用。原函数仍按自己的参数规则绑定；本例 title 仅限位置传入，mark 仅限关键字传入。

In [6]:
def call_once(func):
    """打印进入提示，并把一次调用原样转交给函数。"""

    def wrapper(*args, **kwargs):
        """接收实参并转交调用。"""
        print("进入调用")
        return func(*args, **kwargs)

    return wrapper


@call_once
def format_note(title, /, prefix="笔记", *, mark="。"):
    """把标题格式化为一行笔记。"""
    return f"{prefix}：{title}{mark}"


print(format_note("闭包"))  # 进入调用、笔记：闭包。
print(format_note("包装", prefix="主题", mark="！"))  # 进入调用、主题：包装！

进入调用
笔记：闭包。
进入调用
主题：包装！


In [7]:
# 反例先进入 wrapper，再由原函数的参数绑定引发 TypeError。
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
format_note(title="错误位置")

进入调用


TypeError: format_note() got some positional-only arguments passed as keyword arguments: 'title'

### 4.2 返回原结果，包括 None

包装函数里的 return func(\*args, \*\*kwargs) 把原函数的返回值交回调用方。省略 return 会使包装函数返回 None，即使原函数已经计算出了结果。

返回对象也不需要复制。下面沿用 call\_once，分别检查对象身份和原函数本来就返回 None 的情况。

In [8]:
@call_once
def keep_object(value):
    """直接返回收到的对象。"""
    return value


@call_once
def add_title(titles, title):
    """追加标题，不提供返回值。"""
    titles.append(title)


titles = ["函数"]
print(keep_object(titles) is titles)  # 进入调用、True：返回的是同一列表。
print(add_title(titles, "装饰器"))  # 进入调用、None：保留原函数的返回行为。
print(titles)  # ['函数', '装饰器']：原函数的修改也保留。

进入调用
True
进入调用
None
['函数', '装饰器']


### 4.3 让异常继续传播

原函数引发的异常如果未被包装函数处理，会继续传给外层调用方。为了保持异常行为，不应在包装层捕获所有异常后返回默认值。

原调用后面的普通语句只在调用正常返回时运行。下面沿用 announce，用传入的 ValueError 对象观察异常传播；虽然参数不再是姓名，单个位置实参仍可被该包装函数转交。

In [9]:
@announce
def raise_problem(problem):
    """引发调用方提供的异常对象。"""
    raise problem


problem = ValueError("标题为空")
try:
    raise_problem(problem)
except ValueError as error:
    print(error is problem, str(error))  # True 标题为空：仍是同一个异常。
else:
    raise AssertionError("原异常没有传播到调用方")
# 只出现开始问候，不出现结束问候：异常跳过了调用后的普通语句。

开始问候
True 标题为空


### 4.4 成功和失败都要执行的退出操作

如果退出提示在正常返回和异常传播时都要执行，可以放进 finally。finally 中不要再 return，否则可能替换原返回值或压住原异常；其中的操作若引发新异常，也会影响原来的异常传播。

下面只打印固定提示，不转换异常。这里复用异常处理的规则，用它安排包装函数的退出操作。

In [10]:
def show_exit(func):
    """无论调用成功或失败，都打印退出提示。"""

    def wrapper(*args, **kwargs):
        """保留原调用结果，并在离开时打印提示。"""
        # finally 只记录退出，原返回值或异常继续交给调用方。
        try:
            return func(*args, **kwargs)
        finally:
            print("退出调用")

    return wrapper


@show_exit
def divide(total, count):
    """按给定份数计算每份数量。"""
    return total / count


print(divide(8, 2))  # 退出调用、4.0。

退出调用
4.0


In [11]:
# 失败路径也先输出退出调用，再由调用方处理除零异常。
# 预期 ZeroDivisionError：直接观察原始异常，之后继续运行下一单元。
divide(8, 0)

退出调用


ZeroDivisionError: division by zero

## 5 用 functools.wraps 保留元数据

### 5.1 包装后显示的是谁的信息

元数据（metadata）是名称、文档字符串等描述函数的信息。前面的包装函数虽然能正确转交调用，但它们是独立函数，默认保留自身定义时的名称和说明。

下面沿用 format\_note。查看属性不会调用函数体，也不会打印“进入调用”。

In [12]:
print(format_note.__name__)  # wrapper，而不是 format_note。
print(format_note.__doc__)  # 接收实参并转交调用。

wrapper
接收实参并转交调用。


### 5.2 在包装函数定义处使用 wraps

functools.wraps(func) 用 func 的信息更新包装函数；写在 wrapper 的定义前，得到 @functools.wraps(func)。它是标准库提供的带参数装饰器，下一节再拆解这种结构。

下面列出本章关注的属性。wraps 通过 functools.update\_wrapper 完成更新，不会把包装函数变回原函数。

| 原文名称 | 中文名称／含义 | 默认处理方式 |
| --- | --- | --- |
| \_\_name\_\_ | 函数名称 | 从被包装函数赋值 |
| \_\_qualname\_\_ | 限定名称，包含嵌套位置等信息 | 从被包装函数赋值 |
| \_\_module\_\_ | 定义函数的模块名 | 从被包装函数赋值 |
| \_\_doc\_\_ | 文档字符串 | 从被包装函数赋值 |
| \_\_dict\_\_ | 保存自定义函数属性的字典 | 用被包装函数的属性更新 |
| \_\_wrapped\_\_ | 直接被包装对象的引用 | 自动设置，供后续检查 |

下面的 custom\_note 使用新装饰器；source\_note 是未包装的原函数引用。

In [13]:
import functools


def announce_call(func):
    """保留元数据，并在调用前显示函数名称。"""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        """打印入口名称后转交调用。"""
        print(f"调用 {func.__name__}")
        return func(*args, **kwargs)

    return wrapper


def source_note(title, *, mark="。"):
    """返回带结尾标记的标题。"""
    return title + mark


# 同时观察标准函数元数据与手动添加的属性是否被复制。
source_note.category = "学习"
custom_note = announce_call(source_note)
print(custom_note("装饰器"))  # 调用 source_note、装饰器。
print(custom_note.__name__)  # source_note
print(custom_note.__doc__)  # 返回带结尾标记的标题。
print(custom_note.__qualname__ == source_note.__qualname__)  # True
print(custom_note.__module__ == source_note.__module__)  # True
print(custom_note.category)  # 学习：自定义属性也被带到包装函数上。
print(custom_note is source_note)  # False：元数据相同不代表对象相同。

调用 source_note
装饰器。
source_note
返回带结尾标记的标题。
True
True
学习
False


## 6 带参数装饰器的三层职责

### 6.1 配置、接收函数、处理调用

要让不同函数打印不同标签，可以先传配置，再传函数，最后传调用实参。通常用三层普通函数完成：

| 本例名称 | 中文名称／含义 | 接收内容 | 返回内容 |
| --- | --- | --- | --- |
| tagged | 装饰器工厂 | 非空字符串 label，作为提示标签 | decorate 装饰器 |
| decorate | 装饰器 | 被包装函数 func | wrapper 包装函数 |
| wrapper | 包装函数 | 原函数调用的 args 和 kwargs | 原函数返回值 |

@tagged("学习") 先调用 tagged 得到 decorate，再用 decorate 包装紧随其后的函数。闭包让 wrapper 保留这次配置中的 label 和 func；调用业务函数时不需要再次传入 label。

下面沿用 functools。标签为空字符串时，工厂立即引发 ValueError，无需等到业务函数调用。

In [14]:
def tagged(label):
    """用非空字符串标签创建调用提示装饰器。"""
    if not label:
        raise ValueError("标签不能为空")

    def decorate(func):
        """保存被装饰函数并返回包装函数。"""

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            """显示当前标签并原样转交调用。"""
            print(f"[{label}] {func.__name__}")
            return func(*args, **kwargs)

        return wrapper

    return decorate


@tagged("学习")
def join_title(topic, *, prefix="笔记"):
    """将主题与前缀组成标题。"""
    return f"{prefix}：{topic}"


print(join_title("闭包", prefix="复习"))  # [学习] join_title、复习：闭包。

[学习] join_title
复习：闭包


### 6.2 配置在创建装饰器时传入

带参数形式近似于 join\_title = tagged("学习")(join\_title)。第一次调用取得装饰器，第二次调用接收函数；调用 join\_title 才执行包装后的业务逻辑。

下面沿用 tagged 和 source\_note，把不同的字符串配置用于同一个原函数。每次工厂调用有各自的局部绑定；这不意味着闭包会复制任意可变配置对象。

In [15]:
label = "初读"
first_tag = tagged(label)
label = "复习"
second_tag = tagged(label)

first_note = first_tag(source_note)
second_note = second_tag(source_note)
print(first_note("包装"))  # [初读] source_note、包装。
print(second_note("包装"))  # [复习] source_note、包装。
# 外部 label 重新绑定没有改变先前工厂调用收到的字符串。

[初读] source_note
包装。
[复习] source_note
包装。


In [16]:
# 预期 ValueError：直接观察原始异常，之后继续运行下一单元。
# 标签不能为空：工厂阶段就失败。
tagged("")

ValueError: 标签不能为空

## 7 多个装饰器的顺序

### 7.1 表达式自上而下求值，应用自下而上

叠加装饰器要区分两个阶段：各个 @ 后面的表达式按书写顺序自上而下求值；取得装饰器后，先应用最靠近 def 的那个，再逐层向外应用。

若 outer 和 inner 已经表示两个装饰器，函数名 task 最终近似绑定为 outer(inner(task))。outer 在最外层，inner 接收原函数。

下面 trace\_layer 的 label 标识层次，events 是调用方提供的记录列表。原函数也通过 events 接收记录列表。先只观察定义阶段；finally 中的退出记录在调用阶段才会发生。沿用 functools。

In [17]:
def trace_layer(label, events):
    """记录装饰器的创建、应用和包装函数的进出。"""
    events.append(f"求值 {label}")

    # 外层调用产生装饰器；这一层在把装饰器应用到原函数时运行。
    def decorate(func):
        """记录应用时机，并为函数增加进出记录。"""
        events.append(f"应用 {label}")

        # 只有调用被装饰后的函数时，才进入最内层 wrapper。
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            """记录当前层的进入和退出，保留结果与异常。"""
            events.append(f"进入 {label}")
            try:
                return func(*args, **kwargs)
            finally:
                events.append(f"退出 {label}")

        return wrapper

    return decorate


layer_events = []


@trace_layer("外", layer_events)
@trace_layer("内", layer_events)
def read_item(items, key, events):
    """从字典中读取指定键对应的值。"""
    events.append("原函数")
    return items[key]


print(layer_events)  # ['求值 外', '求值 内', '应用 内', '应用 外']
# 函数尚未调用，记录中没有进入、原函数或退出。

['求值 外', '求值 内', '应用 内', '应用 外']


### 7.2 调用从外向内进入，再从内向外退出

对于每层都调用内层一次的包装函数，调用会先进入最外层，再进入内层，最后到原函数。正常返回时依次经过内层和外层；下面使用 finally，所以异常传播时也按相反方向退出。

这是这些包装函数的控制流产生的顺序。装饰器若提前返回、重复调用或捕获异常，调用过程就会改变，不能只凭 @ 的行数推断。

下面沿用 read\_item 和 layer\_events，先清空定义阶段的记录。

In [18]:
layer_events.clear()
print(read_item({"标题": "装饰器"}, "标题", layer_events))  # 装饰器
print(layer_events)
# ['进入 外', '进入 内', '原函数', '退出 内', '退出 外']

layer_events.clear()
try:
    read_item({}, "标题", layer_events)
except KeyError as error:
    print(error.args)  # ('标题',)：缺失键的异常继续传到调用方。
else:
    raise AssertionError("缺失键没有引发 KeyError")
print(layer_events)
# 失败时仍是进入 外、进入 内、原函数、退出 内、退出 外。

装饰器
['进入 外', '进入 内', '原函数', '退出 内', '退出 外']
('标题',)
['进入 外', '进入 内', '原函数', '退出 内', '退出 外']


## 8 装饰普通实例方法

类中的普通函数经实例访问时会绑定实例，调用时自动把它放在实参最前面。返回普通函数的装饰器可以沿用这种行为：wrapper 的 args 会包含实例，再原样转交给原方法的 self。

下面沿用 announce\_call。NoteBook 是保存前缀的类，render 的 self 代表当前实例；调用 notebook.render 时无需手动补 self。本例只处理普通实例方法。

In [19]:
class NoteBook:
    """保存标题前缀，并提供标题格式化方法。"""

    def __init__(self, prefix):
        self.prefix = prefix

    @announce_call
    def render(self, title, *, mark="。"):
        """结合当前实例的前缀格式化标题。"""
        return f"{self.prefix}：{title}{mark}"


notebook = NoteBook("学习")
review_book = NoteBook("复习")
print(notebook.render("装饰器", mark="！"))  # 调用 render、学习：装饰器！
print(NoteBook.render(notebook, "装饰器"))  # 调用 render、学习：装饰器。
print(review_book.render("闭包"))  # 调用 render、复习：闭包。
# 经类访问时显式提供实例；两种调用都把正确的实例交给原方法。

调用 render
学习：装饰器！
调用 render
学习：装饰器。
调用 render
复习：闭包。


## 9 检查被包装函数与调用签名

### 9.1 用 \_\_wrapped\_\_ 访问直接内层

wraps 设置的 \_\_wrapped\_\_ 指向直接被包装对象。它是检查入口，不会自动调用原函数。显式调用这个属性指向的函数，会绕过当前包装层的操作。

下面沿用 custom\_note 和 source\_note。绕过包装也会绕过其中的检查或记录，使用这个入口时要清楚自己省略了哪些行为。

In [20]:
print(custom_note.__wrapped__ is source_note)  # True
print(custom_note.__wrapped__("原函数", mark="！"))  # 只有原函数！
# 没有“调用 source_note”，因为本次没有进入包装函数。

True
原函数！


### 9.2 叠加时是一条逐层连接的链

每层正确使用 wraps 时，\_\_wrapped\_\_ 只指向紧邻的内层，并不直接跳到最初的函数。inspect.unwrap 则沿着这条属性链找到末端对象。

下面沿用 read\_item 和 layer\_events，比较只去掉外层与找到原函数的区别。

In [21]:
import inspect

inner_read = read_item.__wrapped__
original_read = inspect.unwrap(read_item)
print(inner_read.__wrapped__ is original_read)  # True：本例共有两层包装。

layer_events.clear()
print(inner_read({"标题": "内层"}, "标题", layer_events))  # 内层
print(layer_events)  # ['进入 内', '原函数', '退出 内']

layer_events.clear()
print(original_read({"标题": "原始"}, "标题", layer_events))  # 原始
print(layer_events)  # ['原函数']：原函数自己的行为仍会执行。

True


内层
['进入 内', '原函数', '退出 内']
原始
['原函数']


### 9.3 展示签名与实际包装函数的参数列表

调用签名（call signature）描述函数的参数名称、顺序、默认值和传参限制。inspect.signature 默认沿 \_\_wrapped\_\_ 查找，因此使用 wraps 后通常显示原函数的签名。

传入 follow\_wrapped=False 可以查看当前包装函数本身的签名。元数据使工具能找到原函数的信息，并不把包装函数的 \*args、\*\*kwargs 改写成原函数的参数列表。

下面沿用 custom\_note 和 inspect；显示出的星号表示后续参数仅限关键字传入。

In [22]:
print(inspect.signature(custom_note))  # (title, *, mark='。')
print(inspect.signature(custom_note, follow_wrapped=False))  # (*args, **kwargs)
print(custom_note("标题", mark="！"))  # 调用 source_note、标题！
# 原函数收到解包后的参数，仍按原来的签名检查并执行。

(title, *, mark='。')
(*args, **kwargs)
调用 source_note
标题！


### 9.4 wraps 不会修复错误的参数接收方式

如果 wrapper 漏接关键字参数，wraps 不会自动替它补上。工具显示的原函数签名可能看起来正确，实际调用仍可能在进入包装函数前就失败。

下面故意让 wrapper 只接收位置实参。func 表示被包装函数；普通调用仍返回原结果，带关键字的调用暴露包装层收窄了接口。沿用 functools、inspect 和 source\_note。

In [23]:
def positional_wrapper(func):
    """构造反例：保留元数据，却漏接关键字实参。"""

    @functools.wraps(func)
    def wrapper(*args):
        """只转交位置实参，用于观察接口不一致。"""
        return func(*args)

    return wrapper


limited_note = positional_wrapper(source_note)
print(inspect.signature(limited_note))  # (title, *, mark='。')
print(inspect.signature(limited_note, follow_wrapped=False))  # (*args)
print(limited_note("标题"))  # 标题。

(title, *, mark='。')
(*args)
标题。


In [24]:
# 修复应调整 wrapper 的参数与转发方式，不能只修改显示的元数据。
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
limited_note("标题", mark="！")

TypeError: source_note() got an unexpected keyword argument 'mark'

## 10 wraps 不会消除调用栈帧

栈帧（frame）记录一次函数执行所需的信息。包装函数实际调用原函数时，两者都参与执行；wraps 更新元数据，不会移除这一层调用。

异常的 \_\_traceback\_\_ 保存回溯信息。traceback.extract\_tb 提取其中的帧摘要，每个摘要的 name 来自代码对象中的函数名，不是后来复制到函数对象上的 \_\_name\_\_。因此回溯中仍可能看到 wrapper。

下面沿用 announce\_call，只检查最后两个帧的名称，避免把 Notebook 的临时文件路径当作固定输出。调试时结合回溯和包装链定位，不能把同名元数据当作“没有包装层”。

In [25]:
import traceback


@announce_call
def parse_count(text):
    """把十进制文本转换为整数。"""
    return int(text)


print(parse_count.__name__)  # parse_count：名称已经保留。
try:
    parse_count("不是整数")
except ValueError as error:
    frames = traceback.extract_tb(error.__traceback__)
    print([frame.name for frame in frames[-2:]])  # ['wrapper', 'parse_count']
else:
    raise AssertionError("预期整数转换引发 ValueError")
# 包装层先输出调用 parse_count；wraps 没有隐藏包装函数的执行帧。

parse_count
调用 parse_count
['wrapper', 'parse_count']


## 本章小结

（1）装饰器接收函数并返回新的绑定对象；包装函数通过闭包保存内层函数。带参数形式通常分为配置、装饰、调用三层。

（2）装饰发生在执行定义语句时。叠加表达式自上而下求值，装饰器自下而上应用；本章包装函数的调用从外向内进入，再从内向外退出。

（3）用 \*args、\*\*kwargs 转交参数，用 return 交回结果，让异常自然传播；需要在两种退出路径都执行的操作放进 finally。

（4）wraps 保留元数据并建立 \_\_wrapped\_\_ 链，便于检查签名和定位原函数；它不会修复错误的参数列表，也不会消除栈帧。

自查：能否分别指出“现在创建哪个函数”“这个名称指向谁”“这次调用先进入哪一层”？

## 练习

### 练习 1：分阶段预测记录

沿用 trace\_layer。先不运行，分别写出三个 print 的输出，并区分表达式求值、应用装饰器与执行包装函数的阶段。再运行核对，解释为什么原函数的标记只出现一次。

In [26]:
practice_events = []


@trace_layer("甲", practice_events)
@trace_layer("乙", practice_events)
def practice_total(left, right, events):
    """记录原函数运行并返回两数之和。"""
    events.append("求和")
    return left + right


print(practice_events)  # 分别核对定义阶段与调用阶段的事件次序，先自行预测。
practice_events.clear()
print(practice_total(2, 3, practice_events))
print(practice_events)  # 分别核对定义阶段与调用阶段的事件次序，先自行预测。

['求值 甲', '求值 乙', '应用 乙', '应用 甲']
5
['进入 甲', '进入 乙', '求和', '退出 乙', '退出 甲']


### 练习 2：编写保留调用行为的装饰器

实现 record\_calls(events)，其中 events 是调用方提供的列表。工厂返回装饰器；每次调用只在该列表追加一次被包装函数的名称，然后原样转交调用。使用 functools.wraps 保留元数据。

检查以下情况：一个参数仅限位置、另一个仅限关键字的函数能正常调用；返回列表时保持对象身份；原函数返回 None 时仍为 None；原函数引发给定 ValueError 对象时，调用方收到同一对象。成功和失败调用都应各追加一条记录，原函数每次只执行一次。

In [27]:
# 在此实现 record_calls，并自行定义小函数覆盖题目中的调用情况。
# 只在检查已知失败输入时捕获 ValueError，不在包装层吞掉异常。
pass

### 练习 3：定位并修复方法包装的参数缺口

沿用 positional\_wrapper、functools 和 inspect，新建 PracticeBook 类。初始化时保存 prefix，render(self, title, *, mark="。") 返回前缀、冒号、标题与 mark 组成的字符串，先用 positional\_wrapper 装饰 render。

比较 inspect.signature(PracticeBook.render) 与关闭 follow\_wrapped 后的签名，再用实例调用 render("装饰器", mark="！")，只捕获预期的 TypeError。说明 self 如何到达包装函数，以及 wraps 为什么不能修复这次失败。

另写一个能接收并转发关键字实参的修正版装饰器，重新定义 PracticeBook 后核对：prefix 为“学习”时，上述调用得到“学习：装饰器！”；经类显式传入实例也能调用；名称和文档字符串保留。不要修改前面用于演示反例的 positional\_wrapper。

In [28]:
# 在此实现反例类与修正版；用两种调用方式检查同一个实例。
# 比较签名时同时查看默认结果和 follow_wrapped=False 的结果。
pass

## 参考与引用来源

| 网站 | 已核查的版本、位置与支持内容 |
| --- | --- |
| Python 官方文档（docs.python.org） | Python 3.12：[函数定义](https://docs.python.org/3.12/reference/compound_stmts.html#function-definitions)，支持函数对象、装饰时机、等价绑定、嵌套应用与闭包；[名称解析](https://docs.python.org/3.12/reference/executionmodel.html#resolution-of-names)，支持外层绑定与闭包；[调用](https://docs.python.org/3.12/reference/expressions.html#calls)和[return](https://docs.python.org/3.12/reference/simple_stmts.html#the-return-statement)，支持参数转发、约束与返回语义；[异常处理](https://docs.python.org/3.12/tutorial/errors.html#handling-exceptions)和[finally](https://docs.python.org/3.12/reference/compound_stmts.html#finally-clause)，支持传播、退出次序及覆盖边界；[用户定义函数及属性](https://docs.python.org/3.12/reference/datamodel.html#user-defined-functions)、[update_wrapper](https://docs.python.org/3.12/library/functools.html#functools.update_wrapper)与[wraps](https://docs.python.org/3.12/library/functools.html#functools.wraps)，支持元数据更新和直接包装引用；[方法对象](https://docs.python.org/3.12/tutorial/classes.html#method-objects)，支持 self 的自动传入；[signature](https://docs.python.org/3.12/library/inspect.html#inspect.signature)与[unwrap](https://docs.python.org/3.12/library/inspect.html#inspect.unwrap)，支持签名检查和包装链；[extract_tb](https://docs.python.org/3.12/library/traceback.html#traceback.extract_tb)与[FrameSummary](https://docs.python.org/3.12/library/traceback.html#traceback.FrameSummary)，支持回溯帧与代码名称。 |
| CPython 官方源码（raw.githubusercontent.com） | v3.12.14：[test_decorators.py](https://raw.githubusercontent.com/python/cpython/v3.12.14/Lib/test/test_decorators.py)，test_eval_order 第 233–292 行，尤其第 272–284 行的预期顺序，支持表达式自上而下求值、装饰器自下而上应用；[functools.py](https://raw.githubusercontent.com/python/cpython/v3.12.14/Lib/functools.py)，update_wrapper 与 wraps 第 35–77 行，支持元数据更新及最后设置直接内层引用，未改写包装函数的执行代码。 |